In [70]:
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np

In [71]:
train_df = pd.read_csv('../data/train_cleaned.csv')
test_df = pd.read_csv('../data/test_cleaned.csv')

In [72]:
test_df = test_df.drop('id', axis=1)

In [73]:
Xd = train_df.drop('price_range', axis=1)
yd = train_df['price_range']

DT model

In [ ]:
test_sizes = [0.1, 0.15, 0.2, 0.25, 0.3]
max_depths = range(1, 21)

results = []

for test_size in test_sizes:

    sss = StratifiedShuffleSplit(
        n_splits=10,
        test_size=test_size,
        random_state=42
    )

    for depth in max_depths:

        train_scores = []
        test_scores = []

        for train_idx, test_idx in sss.split(Xd, yd):

            X_train = Xd.iloc[train_idx]
            X_test = Xd.iloc[test_idx]

            y_train = yd.iloc[train_idx]
            y_test = yd.iloc[test_idx]

            clf = DecisionTreeClassifier(
                max_depth=depth,
                random_state=42
            )

            clf.fit(X_train, y_train)

            train_pred = clf.predict(X_train)
            test_pred = clf.predict(X_test)

            train_scores.append(
                accuracy_score(y_train, train_pred)
            )

            test_scores.append(
                accuracy_score(y_test, test_pred)
            )

        results.append({
            'Test_Size': test_size,
            'Max_Depth': depth,
            'Train_Acc_Mean': np.mean(train_scores),
            'Test_Acc_Mean': np.mean(test_scores),
            'Train_Acc_STD': np.std(train_scores),
            'Test_Acc_STD': np.std(test_scores),
            'Gap': np.mean(train_scores) - np.mean(test_scores)
        })

df_results = pd.DataFrame(results)

df_results = df_results.sort_values(
    by='Test_Acc_Mean',
    ascending=False
)

df_results.head(20)

best model is: max depth = 8 and test size = 0.2

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(Xd, yd, test_size=0.2, random_state=42)

best_DT_clf = DecisionTreeClassifier(max_depth=8, random_state=42)
best_DT_clf.fit(X_train, y_train)
print(best_DT_clf.score(X_train, y_train))
print(best_DT_clf.score(X_test, y_test))


check the trained model is predict well.

In [ ]:
y_new_pred = best_DT_clf.predict(test_df)
pd.Series(y_new_pred).value_counts(normalize=True)

In [ ]:
probs = best_DT_clf.predict_proba(test_df)

max_probs = probs.max(axis=1)
print(max_probs.mean())